In [ ]:
#!pip install -U mcp langchain-google-genai langchain-mcp-adapters python-dotenv mysql-connector-python
# mcp dev mcp_server.py

In [1]:
import sys
import os
import io
import asyncio
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient


# Fix Windows asyncio subprocess loop policy
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# Fix Jupyter Notebook (ipykernel) sys.stderr fileno issue on Windows
try:
    import ipykernel.iostream
    ipykernel.iostream.OutStream.fileno = lambda self: sys.__stderr__.fileno() if hasattr(sys.__stderr__, 'fileno') else 2
except Exception:
    pass
try:
    sys.stderr.fileno()
except (io.UnsupportedOperation, AttributeError):
    devnull_fd = os.open(os.devnull, os.O_WRONLY)
    sys.stderr.fileno = lambda: devnull_fd


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

In [4]:
client = MultiServerMCPClient(
    {
        "student_database": {
            "transport": "stdio",
            "command": "python",
            "args": ["mcp_server.py"]
        }
    }
)


In [5]:
tools = await client.get_tools()
# mcp server = tool - list 

In [6]:
print(tools)

[StructuredTool(name='get_student_data', description='Get all student data from MySQL database.', args_schema={'properties': {}, 'title': 'get_student_dataArguments', 'type': 'object'}, handle_tool_error=<function _handle_mcp_tool_error at 0x0000026BA2E358A0>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000026BA2E37100>)]


In [7]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a helpful student assistant.

Use the student database tool whenever
the user asks about student information.

Do not make up student information.
Use the database data to answer.
"""
)

In [8]:
question = input("Ask your question: ")

In [9]:
result = await agent.ainvoke({
    "messages": [
        ("user", question)
    ]
})


In [10]:
print(result["messages"][-1].content[0]["text"])

Based on the student database, here are the details:

### 1. Number of Unique Students
Depending on how "unique" is defined:
* **By Unique ID (Total Records):** There are **39** unique student records (IDs range from 1 to 40, with ID 2 missing).
* **By Unique Student Profiles (excluding ID):** There are **21** unique student profiles, as many records are exact duplicates with different IDs.
* **By Unique Names:** There are **20** unique student names.

---

### 2. Student with the Lengthiest Name
The student with the lengthiest name is **Anjali** (6 letters). 

There are two records for Anjali in the database (which are identical except for their IDs):

#### **Record 1:**
* **ID:** 6
* **Name:** Anjali
* **Fees:** 47,000.00
* **Course:** Python
* **Degree:** BSc
* **City:** Bhavnagar

#### **Record 2:**
* **ID:** 26
* **Name:** Anjali
* **Fees:** 47,000.00
* **Course:** Python
* **Degree:** BSc
* **City:** Bhavnagar
